In [1]:
# pip install python-docx sentence-transformers scikit-learn numpy

In [6]:
# pip install torch

In [7]:
import torch
import torch.nn as nn


In [13]:
# !pip install sentence-transformers

In [11]:
from sentence_transformers import SentenceTransformer
print("Working!")

Working!


In [14]:
# from sentence_transformers import SentenceTransformer
# sentences = ["This is an example sentence", "Each sentence is converted"]

# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
# embeddings = model.encode(sentences)
# print(embeddings)

In [15]:
from sentence_transformers import SentenceTransformer

# Load free embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embedding(text):
    return model.encode(text)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load free embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")


def get_embedding(text):
    return model.encode(text)


def load_text_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()


def chunk_text(text):
    # Split by line and remove empty lines
    chunks = [line.strip() for line in text.split("\n") if line.strip()]
    return chunks


def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    denominator = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    if denominator == 0:
        return 0.0

    return np.dot(vec1, vec2) / denominator


def retrieve_relevant_chunks(question, chunks, top_k=2):
    question_embedding = get_embedding(question)

    scored_chunks = []
    for chunk in chunks:
        chunk_embedding = get_embedding(chunk)
        score = cosine_similarity(question_embedding, chunk_embedding)
        scored_chunks.append((chunk, score))

    scored_chunks.sort(key=lambda x: x[1], reverse=True)
    return scored_chunks[:top_k]




Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
def main():
    file_path = "rag.txt"

    text = load_text_file(file_path)
    chunks = chunk_text(text)

    print("Document loaded.")
    print(f"Total chunks: {len(chunks)}")

    question = input("\nAsk a question: ")

    results = retrieve_relevant_chunks(question, chunks, top_k=4)

    print("\nMost relevant chunks:\n")
    for i, (chunk, score) in enumerate(results, start=1):
        print(f"{i}. Score: {score:.4f}")
        print(chunk)
        print()


if __name__ == "__main__":
    main()

Document loaded.
Total chunks: 29



Ask a question:  who found tesla?



Most relevant chunks:

1. Score: 0.1473
A sustainable business typically has LTV at least 3x CAC.

2. Score: 0.1297
Investors look for strong teams, large markets, and defensible products.

3. Score: 0.1062
- Lifetime Value (LTV)

4. Score: 0.1029
Tracking these metrics helps founders make informed decisions.



# RAG on Research Paper

In [33]:
# !pip install pymupdf

In [38]:
#Read the pdf
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    full_text = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        if text.strip():
            full_text.append(f"\n[Page {page_num}]\n{text}")
    doc.close()
    return "\n".join(full_text)

# Call it like this:
text = extract_text_from_pdf(r"C:\Users\kruta\Desktop\ML PREP\RAG\paperNN.pdf")
print(text[:500])  # Preview first 500 chars


[Page 1]
NEURAL NETWORKS FOR LEARNING MACROSCOPIC
CHEMOTACTIC SENSITIVITY FROM MICROSCOPIC MODELS
RADEK ERBAN ∗
Abstract. The macroscopic (population-level) dynamics of chemotactic cell movement – arising
from underlying microscopic (individual-based) models – are often described by parabolic partial
diﬀerential equations (PDEs) governing the spatio-temporal evolution of cell concentrations.
In
certain cases, these macroscopic PDEs can be analytically derived from microscopic models, thereby
el


In [40]:
#split the paper into chunks
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks


In [42]:
# pip install chromadb

  Using cached chromadb-1.5.5-cp39-abi3-win_amd64.whl.metadata (7.3 kB)
  Using cached uvicorn-0.42.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.40.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached kubernetes-35.0.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached durationpy-0.10-py3-none-any.whl.metadata (340 bytes)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached opentelemetry_exporter_otlp_proto_common-1.40.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached opentelemetry_proto-1.40.0-py3-none-any.whl.metadata (2.4 kB)
  Using ca

In [43]:
#turn chunks into embeddigns
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")


def get_embedding(text: str):
    return model.encode(text).tolist()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
# Store chunks in ChromaDB
def create_vector_store():
    client = chromadb.Client()
    collection = client.get_or_create_collection(name="research_papers")
    return collection


def add_chunks_to_collection(collection, chunks: list[str]):
    for i, chunk in enumerate(chunks):
        collection.add(
            documents=[chunk],
            embeddings=[get_embedding(chunk)],
            ids=[f"chunk_{i}"]
        )

In [45]:
# Retrieve the most relevant chunks
def retrieve_relevant_chunks(collection, question: str, top_k: int = 3):
    query_embedding = get_embedding(question)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    documents = results.get("documents", [[]])[0]
    ids = results.get("ids", [[]])[0]

    return list(zip(ids, documents))